<a href="https://colab.research.google.com/github/Radhika2004Dahiya/complaint-theme-mining/blob/main/complaint-theme-mining_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!git clone https://github.com/Radhika2004Dahiya/complaint-theme-mining.git

Cloning into 'complaint-theme-mining'...
remote: Enumerating objects: 3, done.
remote: Counting objects: 100% (3/3), done.
remote: Total 3 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (3/3), done.


In [3]:
%cd complaint-theme-mining

/content/complaint-theme-mining


In [4]:
!pwd
!ls

/content/complaint-theme-mining
README.md


In [5]:
%cd /content/complaint-theme-mining

/content/complaint-theme-mining


In [6]:
%cd /content/complaint-theme-mining

/content/complaint-theme-mining


In [7]:
!ls

README.md


In [8]:
!pip install pandas numpy scikit-learn sentence-transformers umap-learn hdbscan plotly streamlit -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 49.3 MB/s eta 0:00:00


In [9]:
import sentence_transformers
import hdbscan
import umap
print("All libraries imported successfully")

All libraries imported successfully


In [10]:
!wget https://files.consumerfinance.gov/ccdb/complaints.csv.zip -O complaints.csv.zip

--2026-08-30 11:00:18--  https://files.consumerfinance.gov/ccdb/complaints.csv.zip
Resolving files.consumerfinance.gov (files.consumerfinance.gov)... 23.66.101.44, 23.66.101.46, 2600:1408:ec00:2f::1735:b89, ...
Connecting to files.consumerfinance.gov (files.consumerfinance.gov)|23.66.101.44|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1423555825 (1.3G) [binary/octet-stream]
Saving to: ‘complaints.csv.zip’

complaints.csv.zip  100%[===================>]   1.33G  20.5MB/s    in 74s     

2026-08-30 11:01:32 (18.3 MB/s) - ‘complaints.csv.zip’ saved [1423555825/1423555825]



In [11]:
!unzip complaints.csv.zip
!ls -lh complaints.csv

Archive:  complaints.csv.zip
  inflating: complaints.csv          
-rw-r--r-- 1 root root 8.7G Aug 30 09:18 complaints.csv


In [27]:
import pandas as pd

chunks = []
chunk_size = 200_000

for chunk in pd.read_csv('complaints.csv', chunksize=chunk_size, low_memory=False):
    filtered = chunk[
        (chunk['Product'] == 'Credit card') &
        (chunk['Consumer complaint narrative'].notna())
    ]
    chunks.append(filtered)

df = pd.concat(chunks, ignore_index=True)
print(df.shape)
df.head()

(128109, 16)


,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Submitted via,Date sent to company,Company response to consumer,Timely response?,Complaint ID
0,2026-04-26,Credit card,General-purpose credit card or charge card,Problem with fraud alerts or security freezes,NaN,A collection account from Portfolio Recovery A...,NaN,"Portfolio Recovery Associates, LLC",GA,30087,NaN,Web,2026-06-30,Closed with non-monetary relief,Yes,21613901
1,2026-05-10,Credit card,General-purpose credit card or charge card,Problem with a purchase shown on your statement,Card was charged for something you did not pur...,Company : Chime Financial Inc. Product : Credi...,NaN,Chime Financial Inc,WI,53589,NaN,Web,2026-06-30,Closed with monetary relief,Yes,22059444
2,2026-05-11,Credit card,General-purpose credit card or charge card,"Other features, terms, or problems",Other problem,FORMAL COMPLAINT : SYSTEMIC BILLING FRAUD & DE...,Company has responded to the consumer and the ...,U.S. BANCORP,IL,60622,Older American,Web,2026-06-30,Closed with explanation,Yes,22064282
3,2026-05-11,Credit card,General-purpose credit card or charge card,Problem with a purchase shown on your statement,Card was charged for something you did not pur...,"On XX/XX/year>, fraudulent charges were made o...",NaN,AMERICAN EXPRESS COMPANY,FL,33160,NaN,Web,2026-06-30,Closed with explanation,Yes,22088363
4,2026-05-12,Credit card,General-purpose credit card or charge card,Fees or interest,Problem with fees,This entire situation has caused significant e...,NaN,"Bread Financial Holdings, Inc.",CA,91367,NaN,Web,2026-06-30,Closed with monetary relief,Yes,22097830


In [13]:
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 85.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 70.2 MB/s eta 0:00:00


In [ ]:
import pandas as pd

# Option A: switch to the more forgiving Python engine, skip bad rows
df = pd.read_csv(
    "complaints.csv",
    engine="python",
    on_bad_lines="skip",
    dtype=str,
)

In [1]:
cols = pd.read_csv("complaints.csv", nrows=0).columns.tolist()
print(cols)

NameError: name 'pd' is not defined

In [6]:
usecols = ["Date received", "Product", "Issue", "Consumer complaint narrative", "Company", "State"]

In [7]:
import pandas as pd

chunksize = 100_000
first = True

for chunk in pd.read_csv(
    "complaints.csv",
    engine="python",
    on_bad_lines="skip",
    usecols=usecols,
    dtype=str,
    chunksize=chunksize,
):
    chunk.to_parquet("complaints_clean.parquet", engine="fastparquet", append=not first)
    first = False
    print("wrote chunk")

FileNotFoundError: [Errno 2] No such file or directory: 'complaints.csv'

In [8]:
!pip install fastparquet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 20.4 MB/s eta 0:00:00


In [9]:
import os
if os.path.exists("complaints_clean.parquet"):
    os.remove("complaints_clean.parquet")

In [10]:
import pyarrow.parquet as pq

pf = pq.ParquetFile("complaints_clean.parquet")
print("Rows:", pf.metadata.num_rows)
print("Columns:", pf.schema_arrow)

FileNotFoundError: [Errno 2] Failed to open local file 'complaints_clean.parquet'. Detail: [errno 2] No such file or directory

In [13]:
import os
print(os.getcwd())
print(os.listdir())

/content
['.config', 'complaint-theme-mining', 'sample_data']


In [15]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
!pip install polars
import polars as pl

usecols = ['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue',
           'Consumer complaint narrative', 'Company public response', 'Company',
           'State', 'ZIP code', 'Tags', 'Submitted via', 'Date sent to company',
           'Company response to consumer', 'Timely response?', 'Complaint ID']

df = pl.read_csv(
    "complaints.csv",
    columns=usecols,
    ignore_errors=True,      # skips malformed rows instead of crashing
    infer_schema_length=10000,
    schema_overrides={col: pl.Utf8 for col in usecols},  # force all-string
)

print(df.shape)
df.write_parquet("/content/drive/MyDrive/complaints_clean.parquet")
print("done")

FileNotFoundError: No such file or directory (os error 2): complaints.csv

In [17]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install polars
import polars as pl

usecols = ['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue',
           'Consumer complaint narrative', 'Company public response', 'Company',
           'State', 'ZIP code', 'Tags', 'Submitted via', 'Date sent to company',
           'Company response to consumer', 'Timely response?', 'Complaint ID']

lazy_df = pl.scan_csv(
    "complaints.csv",
    ignore_errors=True,
    infer_schema_length=10000,
    schema_overrides={col: pl.Utf8 for col in usecols},
).select(usecols)

lazy_df.sink_parquet("/content/drive/MyDrive/complaints_clean.parquet")
print("done")

In [18]:
!free -h

               total        used        free      shared  buff/cache   available
Mem:            12Gi       1.2Gi       235Mi       2.0Mi        11Gi        11Gi
Swap:             0B          0B          0B


In [19]:
!head -n 50000 complaints.csv > sample.csv
!ls -lh sample.csv

head: cannot open 'complaints.csv' for reading: No such file or directory
-rw-r--r-- 1 root root 0 Aug 30 09:47 sample.csv


In [ ]:
import polars as pl
import pyarrow as pa
import pyarrow.parquet as pq

usecols = ['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue',
           'Consumer complaint narrative', 'Company public response', 'Company',
           'State', 'ZIP code', 'Tags', 'Submitted via', 'Date sent to company',
           'Company response to consumer', 'Timely response?', 'Complaint ID']

schema = pa.schema([(c, pa.string()) for c in usecols])

reader = pl.read_csv_batched(
    "sample.csv",
    columns=usecols,
    infer_schema_length=1000,
    batch_size=20_000,
)

writer = pq.ParquetWriter("sample_clean.parquet", schema)
total = 0
try:
    while True:
        batches = reader.next_batches(1)
        if not batches:
            break
        for b in batches:
            b = b.select(usecols).cast({c: pl.Utf8 for c in usecols})
            table = b.to_arrow().cast(schema)
            writer.write_table(table)
            total += b.height
        print(f"{total:,} rows written")
finally:
    writer.close()

print("done:", total, "rows")

In [ ]:
!free -h

In [ ]:
import polars as pl
import pyarrow as pa
import pyarrow.parquet as pq

usecols = ['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue',
           'Consumer complaint narrative', 'Company public response', 'Company',
           'State', 'ZIP code', 'Tags', 'Submitted via', 'Date sent to company',
           'Company response to consumer', 'Timely response?', 'Complaint ID']

schema = pa.schema([(c, pa.string()) for c in usecols])
out_path = "/content/drive/MyDrive/complaints_clean.parquet"

reader = pl.read_csv_batched(
    "complaints.csv",
    columns=usecols,
    infer_schema_length=1000,
    batch_size=20_000,
)

writer = pq.ParquetWriter(out_path, schema)
total = 0
batch_count = 0
try:
    while True:
        batches = reader.next_batches(5)
        if not batches:
            break
        for b in batches:
            b = b.select(usecols).cast({c: pl.Utf8 for c in usecols})
            table = b.to_arrow().cast(schema)
            writer.write_table(table)
            total += b.height
            batch_count += 1
        if batch_count % 20 == 0:
            print(f"{total:,} rows written")
finally:
    writer.close()

print("FINAL:", total, "rows written")

In [25]:
import polars as pl
import pyarrow as pa
import pyarrow.parquet as pq

usecols = ['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue',
           'Consumer complaint narrative', 'Company public response', 'Company',
           'State', 'ZIP code', 'Tags', 'Submitted via', 'Date sent to company',
           'Company response to consumer', 'Timely response?', 'Complaint ID']

schema = pa.schema([(c, pa.string()) for c in usecols])
out_path = "/content/drive/MyDrive/complaints_clean.parquet"

reader = pl.read_csv_batched(
    "complaints.csv",
    columns=usecols,
    infer_schema_length=1000,
    batch_size=20_000,
    ignore_errors=True,          # skip rows that fail to parse instead of raising
    truncate_ragged_lines=True,  # tolerate malformed quote/field structure
)

writer = pq.ParquetWriter(out_path, schema)
total = 0
batch_count = 0
try:
    while True:
        batches = reader.next_batches(5)
        if not batches:
            break
        for b in batches:
            b = b.select(usecols).cast({c: pl.Utf8 for c in usecols})
            table = b.to_arrow().cast(schema)
            writer.write_table(table)
            total += b.height
            batch_count += 1
        if batch_count % 20 == 0:
            print(f"{total:,} rows written")
finally:
    writer.close()

print("FINAL:", total, "rows written")

1,050,879 rows written
1,988,854 rows written
2,938,378 rows written
3,809,301 rows written
4,274,837 rows written
4,780,702 rows written
5,311,660 rows written
5,857,873 rows written
6,417,641 rows written
7,000,180 rows written
7,565,014 rows written
8,106,865 rows written
8,703,895 rows written


ComputeError: could not parse `"Date : XX/XX/XXXX Name : XXXX XXXX XXXX XXXX TransUnion, and Equifax Dear Sir or Madam : I am a victim of identity theft. The information listed below, which appears on my credit report, does not relate to any transaction ( s ) that I have made. It is the result of identity theft.

[ Identify item ( s ) resulting from the identity theft that should be blocked, by name of the source, such as the credit card issuer or bank, and type of item, such as credit account, checking account, etc. ] Please block this information from my credit report, pursuant to section 605B of the Fair Credit Reporting Act, and send the required notifications to all furnishers of this information.

The following inquiries and accounts are unauthorized or inaccurate, and I ask that you delete them at once... 

NOTE ... Inquires and accounts please delete ( Experian ) ( Transunion ) and ( Equifax ) please delete these accounts Experian account # XXXX Transunionaccount # XXXX Equifax

In [26]:
import polars as pl

usecols = ['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue',
           'Consumer complaint narrative', 'Company public response', 'Company',
           'State', 'ZIP code', 'Tags', 'Submitted via', 'Date sent to company',
           'Company response to consumer', 'Timely response?', 'Complaint ID']

lazy_df = pl.scan_csv(
    "complaints.csv",
    ignore_errors=True,
    infer_schema_length=1000,
    schema_overrides={c: pl.Utf8 for c in usecols},
).select(usecols)

lazy_df.sink_parquet("/content/drive/MyDrive/complaints_clean.parquet")
print("done")

ComputeError: could not parse `"Date : XX/XX/XXXX Name : XXXX XXXX XXXX XXXX TransUnion, and Equifax Dear Sir or Madam : I am a victim of identity theft. The information listed below, which appears on my credit report, does not relate to any transaction ( s ) that I have made. It is the result of identity theft.

[ Identify item ( s ) resulting from the identity theft that should be blocked, by name of the source, such as the credit card issuer or bank, and type of item, such as credit account, checking account, etc. ] Please block this information from my credit report, pursuant to section 605B of the Fair Credit Reporting Act, and send the required notifications to all furnishers of this information.

The following inquiries and accounts are unauthorized or inaccurate, and I ask that you delete them at once... 

NOTE ... Inquires and accounts please delete ( Experian ) ( Transunion ) and ( Equifax ) please delete these accounts Experian account # XXXX Transunionaccount # XXXX Equifax

In [27]:
import csv
import sys

csv.field_size_limit(sys.maxsize)

usecols = ['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue',
           'Consumer complaint narrative', 'Company public response', 'Company',
           'State', 'ZIP code', 'Tags', 'Submitted via', 'Date sent to company',
           'Company response to consumer', 'Timely response?', 'Complaint ID']

with open("complaints.csv", "r", encoding="utf-8", errors="replace", newline="") as infile, \
     open("complaints_fixed.csv", "w", encoding="utf-8", newline="") as outfile:

    reader = csv.reader(infile)
    header = next(reader)
    col_idx = [header.index(c) for c in usecols]

    writer = csv.writer(outfile)
    writer.writerow(usecols)

    count = 0
    skipped = 0
    for row in reader:
        try:
            if len(row) < len(header):
                skipped += 1
                continue
            writer.writerow([row[i] for i in col_idx])
            count += 1
            if count % 500_000 == 0:
                print(f"{count:,} rows processed")
        except Exception:
            skipped += 1
            continue

print(f"DONE. {count:,} rows written, {skipped:,} rows skipped")

500,000 rows processed
1,000,000 rows processed
1,500,000 rows processed
2,000,000 rows processed
2,500,000 rows processed
3,000,000 rows processed
3,500,000 rows processed
4,000,000 rows processed
4,500,000 rows processed
5,000,000 rows processed
5,500,000 rows processed
6,000,000 rows processed
6,500,000 rows processed
7,000,000 rows processed
7,500,000 rows processed
8,000,000 rows processed
8,500,000 rows processed
9,000,000 rows processed
DONE. 9,287,923 rows written, 1 rows skipped


In [ ]:
import polars as pl

usecols = ['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue',
           'Consumer complaint narrative', 'Company public response', 'Company',
           'State', 'ZIP code', 'Tags', 'Submitted via', 'Date sent to company',
           'Company response to consumer', 'Timely response?', 'Complaint ID']

df = pl.read_csv("complaints_fixed.csv", infer_schema_length=1000, schema_overrides={c: pl.Utf8 for c in usecols})
df.write_parquet("/content/drive/MyDrive/complaints_clean.parquet")
print(df.shape)

In [ ]:
import polars as pl

usecols = ['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue',
           'Consumer complaint narrative', 'Company public response', 'Company',
           'State', 'ZIP code', 'Tags', 'Submitted via', 'Date sent to company',
           'Company response to consumer', 'Timely response?', 'Complaint ID']

lazy_df = pl.scan_csv(
    "complaints_fixed.csv",
    infer_schema_length=1000,
    schema_overrides={c: pl.Utf8 for c in usecols},
)

lazy_df.sink_parquet("/content/drive/MyDrive/complaints_clean.parquet")
print("done")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import polars as pl

usecols = ['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue',
           'Consumer complaint narrative', 'Company public response', 'Company',
           'State', 'ZIP code', 'Tags', 'Submitted via', 'Date sent to company',
           'Company response to consumer', 'Timely response?', 'Complaint ID']

lazy_df = pl.scan_csv(
    "complaints_fixed.csv",
    infer_schema_length=1000,
    schema_overrides={c: pl.Utf8 for c in usecols},
)

lazy_df.sink_parquet("/content/drive/MyDrive/complaints_clean.parquet")
print("done")

In [ ]:
import csv
import sys
import pyarrow as pa
import pyarrow.parquet as pq

csv.field_size_limit(sys.maxsize)

usecols = ['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue',
           'Consumer complaint narrative', 'Company public response', 'Company',
           'State', 'ZIP code', 'Tags', 'Submitted via', 'Date sent to company',
           'Company response to consumer', 'Timely response?', 'Complaint ID']

schema = pa.schema([(c, pa.string()) for c in usecols])
out_path = "/content/drive/MyDrive/complaints_clean.parquet"
batch_size = 20_000

writer = pq.ParquetWriter(out_path, schema)
total = 0

try:
    with open("complaints_fixed.csv", "r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)
        buffer = {c: [] for c in usecols}
        buf_len = 0

        for row in reader:
            for c in usecols:
                buffer[c].append(row.get(c))
            buf_len += 1
            total += 1

            if buf_len >= batch_size:
                table = pa.table(buffer, schema=schema)
                writer.write_table(table)
                buffer = {c: [] for c in usecols}
                buf_len = 0
                if total % 500_000 == 0:
                    print(f"{total:,} rows written")

        # flush remaining rows
        if buf_len > 0:
            table = pa.table(buffer, schema=schema)
            writer.write_table(table)

finally:
    writer.close()

print("FINAL:", total, "rows written")

In [ ]:
!mkdir -p chunks
!head -n 1 complaints_fixed.csv > header.csv
!tail -n +2 complaints_fixed.csv | split -l 300000 -d -a 3 --additional-suffix=.csv - chunks/part_
!ls chunks | wc -l

In [ ]:
import glob, os
import pandas as pd

os.makedirs("/content/drive/MyDrive/complaints_parts", exist_ok=True)
header_text = open("header.csv").read()
parts = sorted(glob.glob("chunks/part_*.csv"))

for i, p in enumerate(parts):
    out_path = f"/content/drive/MyDrive/complaints_parts/part_{i:03d}.parquet"
    if os.path.exists(out_path):
        continue  # already done, skip

    tmp = p + ".full.csv"
    with open(tmp, "w") as out:
        out.write(header_text)
        with open(p) as body:
            out.write(body.read())

    df = pd.read_csv(tmp, dtype=str)
    df.to_parquet(out_path, index=False)
    os.remove(tmp)
    print(f"{i+1}/{len(parts)} done")

print("ALL DONE")

In [ ]:
import csv
import sys
import random
import pandas as pd

csv.field_size_limit(sys.maxsize)

usecols = ['Date received', 'Product', 'Issue', 'Sub-issue', 'Company',
           'State', 'Consumer complaint narrative', 'Submitted via',
           'Company response to consumer', 'Timely response?', 'Complaint ID']

TARGET_TOTAL = 150_000
random.seed(42)

# Pass 1: count rows per Product with a non-null narrative (single fast streaming pass)
from collections import defaultdict
counts = defaultdict(int)

with open("complaints_fixed.csv", "r", encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)
    for row in reader:
        if row.get("Consumer complaint narrative"):
            counts[row["Product"]] += 1

total_with_narrative = sum(counts.values())
print("Rows with narrative:", total_with_narrative)
print(counts)

In [ ]:
import csv, sys, random
import pandas as pd

csv.field_size_limit(sys.maxsize)

usecols = ['Date received', 'Product', 'Issue', 'Sub-issue', 'Company',
           'State', 'Consumer complaint narrative', 'Submitted via',
           'Company response to consumer', 'Timely response?', 'Complaint ID']

# Merge near-duplicate category labels into one canonical name
category_map = {
    'Credit reporting or other personal consumer reports': 'Credit reporting',
    'Credit reporting, credit repair services, or other personal consumer reports': 'Credit reporting',
    'Credit reporting': 'Credit reporting',
    'Credit card or prepaid card': 'Credit card',
    'Credit card': 'Credit card',
    'Prepaid card': 'Credit card',
    'Payday loan, title loan, personal loan, or advance loan': 'Payday/personal loan',
    'Payday loan, title loan, or personal loan': 'Payday/personal loan',
    'Payday loan': 'Payday/personal loan',
    'Consumer Loan': 'Payday/personal loan',
    'Money transfer, virtual currency, or money service': 'Money transfer/virtual currency',
    'Money transfers': 'Money transfer/virtual currency',
    'Virtual currency': 'Money transfer/virtual currency',
    'Bank account or service': 'Checking or savings account',
    'Checking or savings account': 'Checking or savings account',
}

def canon(product):
    return category_map.get(product, product)

TARGET_TOTAL = 150_000
FLOOR = 500        # minimum rows per category (if available)
CAP_FRACTION = 0.30  # no category exceeds 30% of final sample

random.seed(42)

# ---- Pass 1: recompute counts with merged categories ----
from collections import defaultdict
counts = defaultdict(int)
with open("complaints_fixed.csv", "r", encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)
    for row in reader:
        if row.get("Consumer complaint narrative"):
            counts[canon(row["Product"])] += 1

print("Merged categories:")
for k, v in sorted(counts.items(), key=lambda x: -x[1]):
    print(f"  {k}: {v:,}")

# ---- Compute target sample size per category ----
total = sum(counts.values())
cap = int(TARGET_TOTAL * CAP_FRACTION)
raw_targets = {k: max(FLOOR, int(TARGET_TOTAL * v / total)) for k, v in counts.items()}
targets = {k: min(v, cap, counts[k]) for k, v in raw_targets.items()}

print("\nSample targets per category:")
for k, v in sorted(targets.items(), key=lambda x: -x[1]):
    print(f"  {k}: {v:,} (of {counts[k]:,} available)")
print("\nTotal planned sample size:", sum(targets.values()))

In [ ]:
import csv, sys, random
import pyarrow as pa
import pyarrow.parquet as pq
from collections import defaultdict

csv.field_size_limit(sys.maxsize)

usecols = ['Date received', 'Product', 'Issue', 'Sub-issue', 'Company',
           'State', 'Consumer complaint narrative', 'Submitted via',
           'Company response to consumer', 'Timely response?', 'Complaint ID']

category_map = {
    'Credit reporting or other personal consumer reports': 'Credit reporting',
    'Credit reporting, credit repair services, or other personal consumer reports': 'Credit reporting',
    'Credit reporting': 'Credit reporting',
    'Credit card or prepaid card': 'Credit card',
    'Credit card': 'Credit card',
    'Prepaid card': 'Credit card',
    'Payday loan, title loan, personal loan, or advance loan': 'Payday/personal loan',
    'Payday loan, title loan, or personal loan': 'Payday/personal loan',
    'Payday loan': 'Payday/personal loan',
    'Consumer Loan': 'Payday/personal loan',
    'Money transfer, virtual currency, or money service': 'Money transfer/virtual currency',
    'Money transfers': 'Money transfer/virtual currency',
    'Virtual currency': 'Money transfer/virtual currency',
    'Bank account or service': 'Checking or savings account',
    'Checking or savings account': 'Checking or savings account',
}
def canon(product):
    return category_map.get(product, product)

TARGET_TOTAL = 300_000
FLOOR = 800
CAP_FRACTION = 0.30
random.seed(42)

# ---- Pass A: counts ----
counts = defaultdict(int)
with open("complaints_fixed.csv", "r", encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)
    for row in reader:
        if row.get("Consumer complaint narrative"):
            counts[canon(row["Product"])] += 1

total = sum(counts.values())
cap = int(TARGET_TOTAL * CAP_FRACTION)
raw_targets = {k: max(FLOOR, int(TARGET_TOTAL * v / total)) for k, v in counts.items()}
targets = {k: min(v, cap, counts[k]) for k, v in raw_targets.items()}

print("Sample targets per category:")
for k, v in sorted(targets.items(), key=lambda x: -x[1]):
    print(f"  {k}: {v:,} (of {counts[k]:,} available)")
print("\nTotal planned sample size:", sum(targets.values()))

# ---- Pass B: reservoir sampling per category ----
reservoirs = {k: [] for k in targets}
seen = defaultdict(int)

with open("complaints_fixed.csv", "r", encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)
    for row in reader:
        if not row.get("Consumer complaint narrative"):
            continue
        cat = canon(row["Product"])
        if cat not in targets:
            continue
        k = targets[cat]
        seen[cat] += 1
        n = seen[cat]
        record = {c: row.get(c) for c in usecols}
        record["Product_canonical"] = cat
        if len(reservoirs[cat]) < k:
            reservoirs[cat].append(record)
        else:
            j = random.randint(0, n - 1)
            if j < k:
                reservoirs[cat][j] = record

# ---- Combine and write ----
all_rows = [r for rows in reservoirs.values() for r in rows]
random.shuffle(all_rows)
print("\nFinal sample size:", len(all_rows))

cols = usecols + ["Product_canonical"]
table_dict = {c: [r.get(c) for r in all_rows] for c in cols}
table = pa.table(table_dict)
pq.write_table(table, "/content/drive/MyDrive/complaints_sample.parquet")
print("Saved complaints_sample.parquet")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install sentence-transformers umap-learn hdbscan -q